# Import Required Packages

In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
import gensim

In [22]:
from gensim.models import Word2Vec
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split  
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

# Loading and Cleaning Our Data

In [5]:
df0 = pd.read_csv('../data/top_mtsamples.csv')

In [6]:
df1 = df0['transcription']

In [7]:
y = df0['medical_specialty_clean']

In [8]:
review_text = df1.apply(gensim.utils.simple_preprocess)

In [9]:
print(review_text)

0       [mode, left, atrial, enlargement, with, left, ...
1       [the, left, ventricular, cavity, size, and, wa...
2       [echocardiogram, multiple, views, of, the, hea...
3       [description, normal, cardiac, chambers, size,...
4       [study, mild, aortic, stenosis, widely, calcif...
                              ...                        
3099    [indication, chest, pain, type, of, test, aden...
3100    [chief, complaint, chest, pain, history, of, p...
3101    [history, of, present, illness, the, patient, ...
3102    [history, of, present, illness, mr, abc, is, y...
3103    [reason, for, consultation, abnormal, echocard...
Name: transcription, Length: 3104, dtype: object


# Word2Vec Embedding

In [10]:
model = Word2Vec(
    review_text,
    vector_size=100,
    window=10,
    min_count=2
)

In [11]:
#average of the vectors of all words in a document
def document_vector(doc):
    word_vectors = [
        model.wv[word]
        for word in doc
        if word in model.wv
    ]

    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(word_vectors, axis=0)

X = np.array([document_vector(doc) for doc in review_text])

In [12]:
print(X.shape)

(3104, 100)


# Training and Tuning the Model

## LinearSVC

### Training

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

clf = LinearSVC()
clf.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [14]:
#Evaluate Trained Model
y_pred_svc = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_svc))
print(classification_report(y_test,y_pred_svc))

Accuracy: 0.5429184549356223
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.52      0.50      0.51       113
          Gastroenterology       0.44      0.15      0.23        71
          General Medicine       0.58      0.84      0.69        82
                 Neurology       0.49      0.60      0.54        55
   Obstetrics / Gynecology       0.37      0.18      0.24        39
                Orthopedic       0.41      0.27      0.33        96
                 Radiology       0.49      0.29      0.37        85
                   Surgery       0.58      0.79      0.67       346
                   Urology       0.46      0.13      0.21        45

                  accuracy                           0.54       932
                 macro avg       0.48      0.42      0.42       932
              weighted avg       0.52      0.54      0.51       932



### Tuning

In [15]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'loss': ['hinge', 'squared_hinge'],
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(
    estimator=LinearSVC(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro', 
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV score:", grid.best_score_)

/home/addme/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/addme/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/addme/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/addme/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/addme/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/addme/miniforge3/envs/jp/lib/python3.14/site-pack

Best parameters: {'C': 1, 'class_weight': 'balanced', 'loss': 'squared_hinge'}
Best CV score: 0.4795352008107924


In [16]:
best_model = grid.best_estimator_
y_pred_svc = best_model.predict(X_test)
#preds = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_svc))
print(classification_report(y_test,y_pred_svc))

Accuracy: 0.5160944206008584
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.52      0.60      0.56       113
          Gastroenterology       0.53      0.55      0.54        71
          General Medicine       0.58      0.88      0.70        82
                 Neurology       0.45      0.64      0.53        55
   Obstetrics / Gynecology       0.37      0.59      0.46        39
                Orthopedic       0.43      0.58      0.50        96
                 Radiology       0.49      0.35      0.41        85
                   Surgery       0.66      0.40      0.50       346
                   Urology       0.32      0.42      0.37        45

                  accuracy                           0.52       932
                 macro avg       0.48      0.56      0.50       932
              weighted avg       0.54      0.52      0.51       932



## Multinomial Logistic Regression

### Training

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

clf = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

clf.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [18]:
#Predict
y_pred_mlr = clf.predict(X_test)

In [19]:
#Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred_mlr))
print(classification_report(y_test, y_pred_mlr))

Accuracy: 0.5214592274678111
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.51      0.38      0.44       111
          Gastroenterology       0.41      0.25      0.31        67
          General Medicine       0.56      0.71      0.62        78
                 Neurology       0.47      0.60      0.52        67
   Obstetrics / Gynecology       0.42      0.24      0.31        46
                Orthopedic       0.49      0.37      0.42       107
                 Radiology       0.44      0.39      0.41        82
                   Surgery       0.57      0.74      0.65       327
                   Urology       0.32      0.13      0.18        47

                  accuracy                           0.52       932
                 macro avg       0.47      0.42      0.43       932
              weighted avg       0.50      0.52      0.50       932



### Tuning

In [20]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'loss': ['hinge', 'squared_hinge'],
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(
    estimator=LinearSVC(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro', 
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV score:", grid.best_score_)

/home/addme/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/addme/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/addme/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/addme/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/addme/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/addme/miniforge3/envs/jp/lib/python3.14/site-pack

Best parameters: {'C': 0.1, 'class_weight': 'balanced', 'loss': 'squared_hinge'}
Best CV score: 0.48667823500632085


In [21]:
best_model = grid.best_estimator_
y_pred_mlr = best_model.predict(X_test)
#preds = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_mlr))
print(classification_report(y_test,y_pred_mlr))

Accuracy: 0.51931330472103
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.51      0.43      0.47       111
          Gastroenterology       0.46      0.52      0.49        67
          General Medicine       0.55      0.82      0.66        78
                 Neurology       0.51      0.63      0.56        67
   Obstetrics / Gynecology       0.47      0.70      0.56        46
                Orthopedic       0.49      0.58      0.53       107
                 Radiology       0.45      0.39      0.42        82
                   Surgery       0.64      0.46      0.54       327
                   Urology       0.30      0.36      0.33        47

                  accuracy                           0.52       932
                 macro avg       0.49      0.54      0.51       932
              weighted avg       0.53      0.52      0.52       932

